# Qwen3-8B 模型层替换

1. 加载完整 Qwen3-8B；若当前 kernel 已加载同一路径的模型则直接复用；
2. 从完整模型中取得目标层的稠密权重；
3. 执行 TT-matrix SVD，得到内存中的 TT-matrix cores；
4. 运行原始模型，并捕获目标层的真实输入和输出；
5. 用刚分解出的 cores 构造 `TTMatrixLinear`；
6. 替换完整模型中的目标层并再次推理；
7. 对比单层输出、logits、next-token 与生成文本。

模型与 tokenizer 只加载一次。后续可以反复修改 ranks、重新分解和构造 TT-matrix cores，不需要重新加载完整模型。


In [30]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

assert SRC_DIR.is_dir(), f"源码目录不存在：{SRC_DIR}"
print("项目目录：", PROJECT_ROOT)

项目目录： /mnt/intern7/xls/projects/qwen3-tn-compression


In [31]:
import gc
import time

import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer # type: ignore

from qwen3_tn import (
    TTMatrixLinear,
    TTMatrixSpec,
    reconstruct_matrix,
    tt_svd_matrix,
)
from qwen3_tn.experiment import (
    get_module,
    render_prompt,
    replace_module,
    tensor_metrics,
)

print("PyTorch：", torch.__version__)
print("Transformers：", transformers.__version__)
print("CUDA 可用：", torch.cuda.is_available())

PyTorch： 2.5.1
Transformers： 4.51.3
CUDA 可用： True


## 1. 配置目标层与 TT-matrix 结构

默认实验对象是第 0 层 `down_proj`。

In [32]:
MODEL_PATH = Path("/infini-data/Qwen3-8B")
MODULE_PATH = "model.layers.0.mlp.down_proj"
TENSOR_NAME = f"{MODULE_PATH}.weight"

OUT_MODES = (8, 8, 8, 8)
IN_MODES = (8, 8, 8, 24)
TT_RANKS = (1, 64, 512, 192, 1)
SVD_DRIVER = "gesvd"
TOKEN_CHUNK_SIZE = 8
MIN_FREE_GPU_GIB = 60

PROMPT = "请用三句话介绍杭州，并说明最适合游览的季节。"
MAX_NEW_TOKENS = 64

assert MODEL_PATH.is_dir(), f"模型目录不存在：{MODEL_PATH}"
spec = TTMatrixSpec(OUT_MODES, IN_MODES, TT_RANKS)
full_spec = TTMatrixSpec.full_rank(OUT_MODES, IN_MODES)

print("原始模型：", MODEL_PATH)
print("目标权重：", TENSOR_NAME)
print("最大 ranks：", full_spec.ranks)
print("当前 ranks：", spec.ranks)
print("TT 参数量：", f"{spec.num_parameters:,}")
print("稠密参数量：", f"{spec.dense_num_parameters:,}")
print("理论压缩率：", f"{spec.compression_ratio:.2f}x")

原始模型： /infini-data/Qwen3-8B
目标权重： model.layers.0.mlp.down_proj.weight
最大 ranks： (1, 64, 4096, 192, 1)
当前 ranks： (1, 64, 512, 192, 1)
TT 参数量： 8,429,568
稠密参数量： 50,331,648
理论压缩率： 5.97x


## 2. 加载或复用完整 Qwen3-8B

如果当前 kernel 中的 `model` 已经来自同一个 `MODEL_PATH`，这一格不会重新加载权重。若上一轮仍装着 TTMatrixLinear 层，会先恢复原始稠密层。


In [33]:
if not torch.cuda.is_available():
    raise RuntimeError("模型推理与 TT-matrix SVD 需要 CUDA GPU")

device = torch.device("cuda:0")
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision("highest")

resolved_model_path = str(MODEL_PATH.resolve())
existing_model_path = None
if "model" in globals() and isinstance(model, torch.nn.Module):
    existing_model_path = getattr(model, "_qwen3_tn_source_path", None)
    if existing_model_path is None:
        existing_model_path = getattr(
            getattr(model, "config", None),
            "_name_or_path",
            None,
        )

existing_model_path = (
    None
    if existing_model_path is None
    else str(Path(existing_model_path).resolve())
)
model_reused = existing_model_path == resolved_model_path

if not model_reused:
    # 路径变化时先移除旧模型的全部显式引用，避免加载期间同时驻留两个模型。
    old_model = globals().pop("model", None)
    for name in (
        "dense_layer",
        "original_layer",
        "tt_layer",
        "removed_tt_layer",
        "_ACTIVE_TT_REPLACEMENT",
    ):
        globals().pop(name, None)
    del old_model
    gc.collect()
    torch.cuda.empty_cache()

    free_before_load_gib = torch.cuda.mem_get_info(device)[0] / 1024**3
    if free_before_load_gib < 20:
        raise RuntimeError(
            f"加载模型前只有 {free_before_load_gib:.2f} GiB 空闲显存，请先释放其他任务"
        )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        local_files_only=True,
        low_cpu_mem_usage=True,
    )
    model._qwen3_tn_source_path = resolved_model_path
    print("完整模型：本次新加载")
else:
    model._qwen3_tn_source_path = resolved_model_path
    print("完整模型：复用当前 kernel 中已经加载的模型")

model.eval()

tokenizer_reused = (
    "tokenizer" in globals()
    and str(getattr(tokenizer, "name_or_path", "")) == str(MODEL_PATH)
)
if not tokenizer_reused:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        local_files_only=True,
    )
    print("Tokenizer：本次新加载")
else:
    print("Tokenizer：复用当前 kernel 中已经加载的 tokenizer")


def restore_original_layer_if_needed():
    """恢复本 notebook 上一轮安装的 TTMatrixLinear 层；返回被移除的 TTMatrixLinear 层。"""

    state = globals().pop("_ACTIVE_TT_REPLACEMENT", None)
    if state is None or state["model"] is not model:
        return None
    current = get_module(model, state["module_path"])
    if current is state["replacement"]:
        replace_module(model, state["module_path"], state["original"])
        return current
    if current is state["original"]:
        return None
    raise RuntimeError("目标模块已被其他代码修改，无法自动恢复上一轮的原始层")


previous_tt_layer = restore_original_layer_if_needed()
if previous_tt_layer is None:
    # 兼容修改 Notebook 之前已经装入、尚未登记状态的 TTMatrixLinear 层。
    current_target = get_module(model, MODULE_PATH)
    if isinstance(current_target, TTMatrixLinear):
        legacy_original = globals().get("original_layer")
        if legacy_original is None or not hasattr(legacy_original, "weight"):
            raise RuntimeError(
                "检测到未登记的 TT 替换层，但找不到原始稠密层；请重启 kernel 后运行"
            )
        replace_module(model, MODULE_PATH, legacy_original)
        previous_tt_layer = current_target

if previous_tt_layer is not None:
    previous_tt_layer.to("cpu")
    torch.cuda.empty_cache()
    print("已自动恢复上一轮的原始稠密层")

input_device = model.get_input_embeddings().weight.device
dense_layer = get_module(model, MODULE_PATH)
module_device = dense_layer.weight.device
module_dtype = dense_layer.weight.dtype

assert dense_layer.in_features == spec.in_features
assert dense_layer.out_features == spec.out_features
assert module_dtype == torch.bfloat16

print("完整模型输入 device：", input_device)
print("目标模块：", dense_layer)
print("目标模块 device/dtype：", module_device, module_dtype)


完整模型：复用当前 kernel 中已经加载的模型
Tokenizer：复用当前 kernel 中已经加载的 tokenizer
完整模型输入 device： cuda:0
目标模块： Linear(in_features=12288, out_features=4096, bias=False)
目标模块 device/dtype： cuda:0 torch.bfloat16


## 3. 检查模型加载后的可用显存

这里检查的是完整模型已经驻留 GPU 后，剩余显存是否足以执行 TT-matrix SVD。


In [34]:
free_bytes, total_bytes = torch.cuda.mem_get_info(device)
free_gib = free_bytes / 1024**3
total_gib = total_bytes / 1024**3

print("GPU：", torch.cuda.get_device_name(device))
print(f"模型加载后空闲/总显存：{free_gib:.2f}/{total_gib:.2f} GiB")
print(f"PyTorch allocated：{torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"PyTorch reserved：{torch.cuda.memory_reserved() / 1024**3:.2f} GiB")

if free_gib < MIN_FREE_GPU_GIB:
    raise RuntimeError(
        f"模型加载后空闲显存不足：TT-SVD 至少需要 {MIN_FREE_GPU_GIB} GiB，"
        f"当前 {free_gib:.2f} GiB"
    )


GPU： NVIDIA A100-SXM4-80GB
模型加载后空闲/总显存：63.16/79.15 GiB
PyTorch allocated：15.26 GiB
PyTorch reserved：15.45 GiB


## 4. 从已加载模型取得目标稠密矩阵

直接读取 `dense_layer.weight` 并创建 FP32 分解副本，不再从 Safetensors 重复加载该权重。


In [35]:
# 如果只重跑后续单元，确保模型中安装的是原始稠密层。
previous_tt_layer = restore_original_layer_if_needed()
if previous_tt_layer is not None:
    previous_tt_layer.to("cpu")
    torch.cuda.empty_cache()

dense_layer = get_module(model, MODULE_PATH)
module_device = dense_layer.weight.device
module_dtype = dense_layer.weight.dtype

assert tuple(dense_layer.weight.shape) == (spec.out_features, spec.in_features)
print("稠密权重形状：", tuple(dense_layer.weight.shape))
print("模型权重 dtype/device：", dense_layer.weight.dtype, dense_layer.weight.device)

weight_fp32 = dense_layer.weight.detach().to(device=device, dtype=torch.float32)
print("分解用权重：", weight_fp32.dtype, weight_fp32.device)


稠密权重形状： (4096, 12288)
模型权重 dtype/device： torch.bfloat16 cuda:0
分解用权重： torch.float32 cuda:0


## 5. TT-matrix SVD


In [36]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device)
torch.cuda.synchronize(device)
started = time.perf_counter()

cores_fp32 = tt_svd_matrix(
    weight_fp32,
    spec,
    svd_driver=SVD_DRIVER,
)

torch.cuda.synchronize(device)
svd_elapsed = time.perf_counter() - started
svd_peak_gib = torch.cuda.max_memory_allocated(device) / 1024**3

print(f"TT-matrix SVD 耗时：{svd_elapsed:.2f} 秒")
print(f"TT-matrix SVD 峰值分配显存：{svd_peak_gib:.2f} GiB")
for index, core in enumerate(cores_fp32):
    print(f"core {index}: {tuple(core.shape)}, {core.dtype}, {core.device}")

TT-matrix SVD 耗时：6.93 秒
TT-matrix SVD 峰值分配显存：16.83 GiB
core 0: (1, 8, 8, 64), torch.float32, cuda:0
core 1: (64, 8, 8, 512), torch.float32, cuda:0
core 2: (512, 8, 8, 192), torch.float32, cuda:0
core 3: (192, 8, 24, 1), torch.float32, cuda:0


## 6. 检查分解误差，并准备 BF16 TT-matrix cores

先重构一次稠密矩阵用于诊断。随后把 cores 转成 BF16 并移到 CPU，并释放 SVD 阶段的 FP32 临时张量；已经加载的完整模型保持不变。


In [37]:
with torch.inference_mode():
    reconstructed_fp32 = reconstruct_matrix(cores_fp32, spec)
    weight_comparison = tensor_metrics(weight_fp32, reconstructed_fp32)

print("稠密权重 vs TT 重构权重：")
for key, value in weight_comparison.items():
    print(f"  {key}: {value}")

cores_bf16_cpu = [
    core.to(device="cpu", dtype=torch.bfloat16).contiguous()
    for core in cores_fp32
]

del reconstructed_fp32, cores_fp32, weight_fp32
torch.cuda.empty_cache()
free_after_svd_gib = torch.cuda.mem_get_info(device)[0] / 1024**3

print("TT-matrix cores 已转为 BF16 并移到 CPU")
print(f"释放 SVD 张量后空闲显存：{free_after_svd_gib:.2f} GiB")

稠密权重 vs TT 重构权重：
  relative_l2: 0.8612583194660326
  max_abs: 0.4797780513763428
  cosine: 0.5081674007160958
  finite: True
TT-matrix cores 已转为 BF16 并移到 CPU
释放 SVD 张量后空闲显存：63.16 GiB


## 7. 准备 prompt，并运行原始稠密模型

第一次前向同时捕获目标层的真实输入和输出；hook 随后立即移除，不参与逐 token 生成。


In [38]:
rendered_prompt = render_prompt(tokenizer, PROMPT)
encoded = tokenizer([rendered_prompt], return_tensors="pt").to(input_device)
prompt_token_count = int(encoded.input_ids.shape[1])

print("Prompt：", PROMPT)
print("Prompt token 数：", prompt_token_count)

Prompt： 请用三句话介绍杭州，并说明最适合游览的季节。
Prompt token 数： 25


In [39]:
previous_tt_layer = restore_original_layer_if_needed()
if previous_tt_layer is not None:
    previous_tt_layer.to("cpu")
    torch.cuda.empty_cache()

dense_layer = get_module(model, MODULE_PATH)
module_device = dense_layer.weight.device
module_dtype = dense_layer.weight.dtype

captured = {}

def capture_input(_module, module_inputs):
    captured["input"] = module_inputs[0].detach().cpu()

def capture_output(_module, _module_inputs, module_output):
    captured["output"] = module_output.detach().cpu()

pre_handle = dense_layer.register_forward_pre_hook(capture_input)
output_handle = dense_layer.register_forward_hook(capture_output)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()
try:
    with torch.inference_mode():
        dense_forward = model(**encoded, use_cache=False)
finally:
    pre_handle.remove()
    output_handle.remove()

with torch.inference_mode():
    dense_generated = model.generate(
        **encoded,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
dense_elapsed = time.perf_counter() - started
dense_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

dense_logits = dense_forward.logits[0, -1].float().cpu()
dense_token_ids = dense_generated[0, prompt_token_count:].cpu()
dense_response = tokenizer.decode(dense_token_ids, skip_special_tokens=True)
del dense_forward, dense_generated

print(f"原始模型推理耗时：{dense_elapsed:.2f} 秒")
print(f"原始模型峰值分配显存：{dense_peak_gib:.2f} GiB")
print("原始 next-token：", int(dense_logits.argmax().item()))
print("原始回答：")
print(dense_response)

原始模型推理耗时：1.81 秒
原始模型峰值分配显存：15.29 GiB
原始 next-token： 104130
原始回答：
杭州，被誉为“人间天堂”，以西湖美景、灵隐古刹和龙井茶香闻名于世，是江南水乡的代表。这里四季分明，气候宜人，尤其适合在春秋季游览。春天，西湖边的桃花与柳树交织成画；秋天，满城桂花


## 8. 用刚刚分解出的 cores 构造 TTMatrixLinear 层

这一步先用真实激活做单层比较，尚未修改完整模型。

In [40]:
previous_tt_layer = restore_original_layer_if_needed()
if previous_tt_layer is not None:
    previous_tt_layer.to("cpu")
    torch.cuda.empty_cache()

dense_layer = get_module(model, MODULE_PATH)
module_device = dense_layer.weight.device

tt_layer = TTMatrixLinear(
    spec,
    cores_bf16_cpu,
    token_chunk_size=TOKEN_CHUNK_SIZE,
    trainable=False,
    preserve_input_dtype=True,
).to(module_device)
tt_layer.eval()

with torch.inference_mode():
    tt_layer_output = tt_layer(captured["input"].to(module_device)).float().cpu()

layer_comparison = tensor_metrics(
    captured["output"].float(),
    tt_layer_output,
)

print("原始层输出 vs TTMatrixLinear 层输出：")
for key, value in layer_comparison.items():
    print(f"  {key}: {value}")

原始层输出 vs TTMatrixLinear 层输出：
  relative_l2: 0.7613660367826912
  max_abs: 6.71875
  cosine: 0.7062511576649885
  finite: True


## 9. 替换完整模型中的目标层

In [41]:
previous_tt_layer = restore_original_layer_if_needed()
if previous_tt_layer is not None:
    previous_tt_layer.to("cpu")
    torch.cuda.empty_cache()

dense_layer = get_module(model, MODULE_PATH)
module_device = dense_layer.weight.device
tt_layer.to(module_device)
original_layer = replace_module(model, MODULE_PATH, tt_layer)
assert original_layer is dense_layer
assert get_module(model, MODULE_PATH) is tt_layer

_ACTIVE_TT_REPLACEMENT = {
    "model": model,
    "module_path": MODULE_PATH,
    "original": original_layer,
    "replacement": tt_layer,
}

print("替换完成：", MODULE_PATH)
print("当前模块：", get_module(model, MODULE_PATH))


替换完成： model.layers.0.mlp.down_proj
当前模块： TTMatrixLinear(
  (cores): ParameterList(
      (0): Parameter containing: [torch.bfloat16 of size 1x8x8x64 (cuda:0)]
      (1): Parameter containing: [torch.bfloat16 of size 64x8x8x512 (cuda:0)]
      (2): Parameter containing: [torch.bfloat16 of size 512x8x8x192 (cuda:0)]
      (3): Parameter containing: [torch.bfloat16 of size 192x8x24x1 (cuda:0)]
  )
)


## 10. 使用替换后的完整模型推理

In [42]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
started = time.perf_counter()

with torch.inference_mode():
    tt_forward = model(**encoded, use_cache=False)
    tt_generated = model.generate(
        **encoded,
        do_sample=False,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )

torch.cuda.synchronize()
tt_elapsed = time.perf_counter() - started
tt_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

tt_logits = tt_forward.logits[0, -1].float().cpu()
tt_token_ids = tt_generated[0, prompt_token_count:].cpu()
tt_response = tokenizer.decode(tt_token_ids, skip_special_tokens=True)
del tt_forward, tt_generated

print(f"替换后推理耗时：{tt_elapsed:.2f} 秒")
print(f"替换后峰值分配显存：{tt_peak_gib:.2f} GiB")
print("替换后 next-token：", int(tt_logits.argmax().item()))
print("替换后回答：")
print(tt_response)

替换后推理耗时：1.83 秒
替换后峰值分配显存：15.36 GiB
替换后 next-token： 104130
替换后回答：
杭州，被誉为“人间天堂”，以西湖、灵隐寺、龙井茶等自然与人文景观闻名，是江南水乡与文化古韵的完美结合。四季皆美，但春秋季最佳，春有桃花、秋有桂花，气候宜人，景色如画。若想感受


## 11. 汇总对比

In [43]:
logits_comparison = tensor_metrics(dense_logits, tt_logits)
dense_top1 = int(dense_logits.argmax().item())
tt_top1 = int(tt_logits.argmax().item())

common_prefix_tokens = 0
for dense_id, tt_id in zip(dense_token_ids.tolist(), tt_token_ids.tolist()):
    if dense_id != tt_id:
        break
    common_prefix_tokens += 1

comparison = {
    "weight_relative_l2": weight_comparison["relative_l2"],
    "layer_output_relative_l2": layer_comparison["relative_l2"],
    "logits_relative_l2": logits_comparison["relative_l2"],
    "logits_cosine": logits_comparison["cosine"],
    "next_token_match": dense_top1 == tt_top1,
    "generation_exact_match": torch.equal(dense_token_ids, tt_token_ids),
    "common_prefix_tokens": common_prefix_tokens,
    "dense_seconds": dense_elapsed,
    "tt_seconds": tt_elapsed,
}

for key, value in comparison.items():
    print(f"{key}: {value}")

print("\n--- 原始回答 ---")
print(dense_response)
print("\n--- 替换后回答 ---")
print(tt_response)

weight_relative_l2: 0.8612583194660326
layer_output_relative_l2: 0.7613660367826912
logits_relative_l2: 0.3373946921237109
logits_cosine: 0.9628834954395702
next_token_match: True
generation_exact_match: False
common_prefix_tokens: 9
dense_seconds: 1.8144522930961102
tt_seconds: 1.826755807036534

--- 原始回答 ---
杭州，被誉为“人间天堂”，以西湖美景、灵隐古刹和龙井茶香闻名于世，是江南水乡的代表。这里四季分明，气候宜人，尤其适合在春秋季游览。春天，西湖边的桃花与柳树交织成画；秋天，满城桂花

--- 替换后回答 ---
杭州，被誉为“人间天堂”，以西湖、灵隐寺、龙井茶等自然与人文景观闻名，是江南水乡与文化古韵的完美结合。四季皆美，但春秋季最佳，春有桃花、秋有桂花，气候宜人，景色如画。若想感受


## 12. 可选：恢复原始层

In [44]:
removed_tt_layer = restore_original_layer_if_needed()
if removed_tt_layer is None:
    print("当前已经是原始稠密层，无需恢复。")
else:
    removed_tt_layer.to("cpu")
    torch.cuda.empty_cache()
    print("原始稠密层已恢复，现场分解得到的 TTMatrixLinear 层已移到 CPU。")


原始稠密层已恢复，现场分解得到的 TTMatrixLinear 层已移到 CPU。
